In [2]:
%reload_ext dotenv
%dotenv
from massive import RESTClient
from os import environ
import pandas as pd
from pathlib import Path
from collections.abc import Generator
from datetime import date
from urllib3.exceptions import MaxRetryError
from time import sleep
from math import ceil
from random import randint

In [3]:
# Requires API key in .env file or added to environment
client = RESTClient(api_key=environ["MASSIVE_KEY"])

In [4]:
# Loads data for a single ticker over some period
# This does notably include after-hours trading, which we may want to filter out
# Rate limit is 5 internal requests per minute, which translates to about 1 ticker-month per minute
def get_stock_data(ticker: str, start_date: str, end_date: str, steps: list[int] = list(range(1,21)) + [60]) -> Generator[tuple[date, pd.DataFrame]]:
    while True:
        try:
            aggs = list(client.list_aggs(ticker, 1, "minute", start_date, end_date))
            break
        except MaxRetryError:
            print(f"Rate limited, retrying in 1 minute")
            inc = 10
            for i in range(ceil(60/inc)):
                sleep(inc)
                print(f"{60-(i+1)*inc} seconds remaining")

    data = pd.DataFrame(aggs)

    # Adds time column (EST, in alignment with actual stock market) for convenience
    data["time"] = pd.to_datetime(data["timestamp"], unit='ms').dt.tz_localize("UTC").dt.tz_convert("US/Eastern")

    for (day, day_data) in data.groupby(data["time"].dt.date):
        # Formats data (collects open/time parameters, adds stepped columns to look back in time according to steps list)
        f_data = day_data[["open", "time"]]
        # for step in steps:
        #     f_data[f"back_{step}"] = f_data["open"].shift(step)
        # f_data = f_data.iloc[max(steps):] # clear out NaNs created by steps
        yield (day,f_data)


In [5]:
def pick_month() -> str:
    year = randint(2025, 2026)
    month = randint(1, 12)
    if month >= 6:
        year -= 1
    return f"{year:04d}-{month:02d}"

def pick_months(num_months: int) -> set[str]:
    months = set()
    while len(months) < num_months:
        months.add(pick_month())
    return months

In [6]:
tickers = [
    "AAPL", "ADBE", "MSFT", "GOOG", "AMZN",
    "TSLA", "NVDA", "META", "AMD", "INTC",
    "MU", "ARM", "TXN", "QCOM", "STX",
    "WDC", "SHOP", "ORCL", "AVGO", "CRWD",
    "RDDT"
]

user = 0
num_users = 3
months_per_ticker = 3
for ticker in tickers[user::num_users]:
    print(f"Getting data for {ticker}")
    Path(f"data/{ticker}").mkdir(parents=True, exist_ok=True)
    for month in pick_months(months_per_ticker):
        print(f"Getting {month}")
        for (day, data) in get_stock_data(ticker, f"{month}-01", f"{month}-28"):
            data.to_parquet(f"data/{ticker}/{day}.parquet.gz", compression="gzip")

Getting data for AAPL
Getting 2026-03


Getting 2025-11
Rate limited, retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting 2024-09
Getting data for GOOG
Getting 2025-09
Getting 2025-12
Rate limited, retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting 2025-11
Rate limited, retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting data for NVDA
Getting 2025-09
Rate limited, retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting 2025-06
Rate limited, retrying in 1 minute
50 seconds remaining
40 seconds remaining
30 seconds remaining
20 seconds remaining
10 seconds remaining
0 seconds remaining
Getting 2024-06
Rate limited, retrying in 1 